# INSIVUMEH | Visualizador de imágenes Landsat 8 y 9

**Práctica: composición de bandas, índices espectrales y calidad de observaciones**

### Objetivo

Consultar escenas Landsat 8 y 9, representar distintas combinaciones espectrales y analizar qué información aporta cada canal. El cuaderno permite delimitar un área de estudio, elegir un período de adquisición, generar mapas georreferenciados y descargar los productos obtenidos.

### Funcionamiento

1. Definir el área y el período de consulta.
2. Buscar y seleccionar una escena Landsat 8 o 9.
3. Elegir las visualizaciones y generar la galería.
4. Examinar los mapas y las **fichas de lectura espectral**, situadas debajo de cada imagen.
5. Descargar los GeoTIFF, PNG, fichas y metadatos.

La región inicial corresponde al **volcán de Fuego, Guatemala**. El análisis puede trasladarse a otras áreas con las coordenadas y fechas disponibles.

**Criterio de lectura:** un color de pantalla (R, G o B) puede representar una banda visible o infrarroja. En un falso color, un objeto rojo no necesariamente refleja luz roja visible; puede reflejar intensamente la banda asignada al canal R.

In [ ]:
%pip -q install pystac-client planetary-computer rasterio ipywidgets matplotlib pandas numpy

In [ ]:
import os, math, ast, json, warnings, zipfile
from datetime import date, timedelta
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import rasterio
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.transform import from_bounds
from rasterio.crs import CRS
from pystac_client import Client
import planetary_computer
import ipywidgets as W
from IPython.display import display, clear_output, FileLink, HTML
from concurrent.futures import ThreadPoolExecutor
OUT=Path('/content/landsat_insivumeh') if Path('/content').exists() else Path.cwd()/'landsat_insivumeh'
OUT.mkdir(parents=True,exist_ok=True)
VOLCANES={'Fuego':(14.473,-90.880),'Santiaguito':(14.756,-91.552),'Pacaya':(14.381,-90.601),'Coordenadas personalizadas':(14.473,-90.880)}
BANDS={'B2':'blue','B3':'green','B4':'red','B5':'nir08','B6':'swir16','B7':'swir22'}
PRESETS={
 'True color (B4-B3-B2)':('rgb',['B4','B3','B2']),
 'False color (B5-B4-B3)':('rgb',['B5','B4','B3']),
 'Optimized natural color (B4-B3-B2)':('rgb',['B4','B3','B2']),
 'NDVI':('index','NDVI'),
 'Wildfires / SWIR (B7-B6-B4)':('rgb',['B7','B6','B4']),
 'False color urban (B7-B6-B4)':('rgb',['B7','B6','B4']),
 'Moisture index / NDMI':('index','NDMI'),
 'SWIR (B7-B6-B4)':('rgb',['B7','B6','B4']),
 'NDWI (Green-NIR)':('index','NDWI'),
 'NDSI (Green-SWIR1)':('index','NDSI'),
 'Calidad del píxel (QA_PIXEL)':('qa','QA_PIXEL'),
 'Custom RGB':('custom_rgb',None),
 'Custom formula':('custom_formula',None),
 'Surface temperature °C (ST_B10)':('temp',None),
}
print('Dependencias y carpeta listas:',OUT)

Dependencias y carpeta listas: /content/landsat_insivumeh


## 1. Área de estudio y período de consulta

Seleccionar el volcán o introducir coordenadas en grados decimales. Definir las fechas de inicio y final; el botón **Usar ±7 días** establece un intervalo de dos semanas alrededor de la fecha central. El radio determina la extensión del recorte.

In [ ]:
hoy=date.today()
volcan=W.Dropdown(options=list(VOLCANES),value='Fuego',description='Volcán:',layout=W.Layout(width='370px'))
lat=W.FloatText(value=14.473,description='Latitud:',layout=W.Layout(width='240px'))
lon=W.FloatText(value=-90.880,description='Longitud:',layout=W.Layout(width='260px'))
radio=W.FloatSlider(value=8,min=1,max=20,step=0.5,description='Radio km:',readout_format='.1f',layout=W.Layout(width='400px'))
inicio=W.DatePicker(value=hoy-timedelta(days=14),description='Inicio:')
fin=W.DatePicker(value=hoy,description='Final:')
centro=W.DatePicker(value=hoy-timedelta(days=7),description='Fecha centro:')
centrar=W.Button(description='Aplicar ±7 días',button_style='info')
status=W.HTML()
def cambiar_volcan(change):
    la,lo=VOLCANES[change['new']];lat.value=la;lon.value=lo
volcan.observe(cambiar_volcan,names='value')
def set_14(_):
    if centro.value: inicio.value=centro.value-timedelta(days=7);fin.value=centro.value+timedelta(days=7)
centrar.on_click(set_14)
display(W.VBox([W.HBox([volcan,radio]),W.HBox([lat,lon]),W.HBox([inicio,fin]),W.HBox([centro,centrar]),status]))

## 2. Búsqueda de escenas

El botón **Buscar escenas** consulta las adquisiciones Landsat 8 y 9 para la zona y el período definidos. La tabla muestra fecha UTC, satélite, nubosidad de la escena completa e identificador del producto. Seleccionar una escena para continuar.

La nubosidad del catálogo corresponde a la escena completa y puede diferir de la nubosidad observada dentro del recorte.

In [ ]:
escenas=[]
selector=W.Dropdown(options=[],description='Escena:',layout=W.Layout(width='950px'))
orden=W.Dropdown(options=[('Menor nubosidad','cloud'),('Más reciente','recent')],value='cloud',description='Orden:')
buscar=W.Button(description='Buscar escenas',button_style='primary')
res_busqueda=W.Output()
def bbox_area(la,lo,rkm):
    dlat=rkm/111.32;dlon=rkm/(111.32*max(.1,math.cos(math.radians(la))))
    return (lo-dlon,la-dlat,lo+dlon,la+dlat)
def ejecutar_busqueda(_):
    global escenas
    with res_busqueda:
        clear_output(wait=True)
        try:
            assert inicio.value and fin.value and inicio.value<=fin.value,'Intervalo de fechas inválido'
            assert -90<=lat.value<=90 and -180<=lon.value<=180,'Coordenadas inválidas'
            print('Buscando escenas Landsat...')
            catalog=Client.open('https://planetarycomputer.microsoft.com/api/stac/v1')
            q=catalog.search(collections=['landsat-c2-l2'],bbox=bbox_area(lat.value,lon.value,radio.value),datetime=f'{inicio.value.isoformat()}T00:00:00Z/{fin.value.isoformat()}T23:59:59Z',limit=300)
            escenas=list(q.items())
            escenas.sort(key=(lambda x:float(x.properties.get('eo:cloud_cover',100) or 100)) if orden.value=='cloud' else (lambda x:x.datetime),reverse=orden.value=='recent')
            selector.options=[(f"{x.datetime:%Y-%m-%d %H:%M} UTC | {x.properties.get('platform','?')} | nubes {x.properties.get('eo:cloud_cover',float('nan')):.1f}% | {x.id}",x.id) for x in escenas]
            display(pd.DataFrame([{'fecha UTC':x.datetime.isoformat(),'satélite':x.properties.get('platform'),'nubes escena %':x.properties.get('eo:cloud_cover'),'id':x.id} for x in escenas]).head(100))
            print(f'{len(escenas)} escenas encontradas. Elige una escena en el selector inferior.')
        except Exception as e:print('Error al buscar:',repr(e))
buscar.on_click(ejecutar_busqueda)
display(W.HBox([orden,buscar]),res_busqueda,selector)

Output()

Dropdown(description='Escena:', layout=Layout(width='950px'), options=(), value=None)

## 3. Visualizaciones disponibles

Los mapas se generan a partir de las bandas de reflectancia superficial (Collection 2 Level-2), excepto la clasificación QA_PIXEL y la temperatura superficial. **Cada imagen incluirá una ficha que explica sus canales, rangos espectrales y rasgos destacados.**

| Visualización | Combinación o cálculo | Lectura principal |
|---|---|---|
| True color | R=B4, G=B3, B=B2 | Apariencia semejante al color natural; nubes, vegetación y terreno |
| False color | R=B5, G=B4, B=B3 | La vegetación sana suele aparecer roja por la alta reflectancia NIR |
| Optimized natural color | R=B4, G=B3, B=B2 | **Mismas bandas que True Color**, con un ajuste visual de gamma |
| Wildfires / SWIR | R=B7, G=B6, B=B4 | Contrastes de materiales, humedad y posibles fuentes térmicas intensas |
| False color urban | R=B7, G=B6, B=B4 | **Mismo RGB que Wildfires / SWIR**, interpretado para materiales y uso urbano |
| SWIR | R=B7, G=B6, B=B4 | **Mismo RGB**; se utiliza para destacar respuesta SWIR |
| NDVI | (B5 − B4) / (B5 + B4) | Vegetación fotosintéticamente activa y contraste con superficies desnudas |
| NDMI | (B5 − B6) / (B5 + B6) | Humedad relativa de la vegetación |
| NDWI (Green–NIR) | (B3 − B5) / (B3 + B5) | Presencia potencial de agua superficial; no equivale al NDMI |
| NDSI | (B3 − B6) / (B3 + B6) | Nieve/hielo potencial, sujeto a máscaras y contexto |
| QA_PIXEL | Indicadores binarios | Nubes, sombras, cirros, agua y datos ausentes; **no es clasificación de uso del suelo** |
| Custom RGB | R, G y B seleccionables | Permite estudiar cómo cambia el color aparente al reasignar bandas |
| Custom formula | Expresión matemática | Ejemplo inicial: diferencia normalizada entre B7 y B6; no es temperatura |
| ST_B10 | Producto de temperatura superficial | Temperatura de superficie en °C; dato térmico independiente de RGB |

**Importante:** Wildfires / SWIR, False color urban y SWIR presentan imágenes idénticas si usan el mismo estiramiento. Son nombres de interpretación, no algoritmos distintos ni detectores automáticos de incendios.

### Bandas Landsat 8/9: longitudes de onda y aplicaciones

Las longitudes de onda se expresan en **micrómetros (µm)**. Un micrómetro equivale a **1 000 nanómetros (nm)**. Los intervalos son bandas espectrales aproximadas del instrumento, no una única longitud de onda.

| Banda | Región espectral | Longitud de onda | Resolución nativa | Qué ayuda a distinguir |
|---|---|---|---:|---|
| B1 | Costera / aerosol | 0.43–0.45 µm | 30 m | Aerosoles y propiedades ópticas del agua |
| **B2** | **Azul visible** | **0.45–0.51 µm** | **30 m** | Agua, dispersión atmosférica y contrastes en superficies claras |
| **B3** | **Verde visible** | **0.53–0.59 µm** | **30 m** | Reflectancia verde de la vegetación y características ópticas del agua |
| **B4** | **Rojo visible** | **0.64–0.67 µm** | **30 m** | Absorción de clorofila y diferencias entre suelo y vegetación |
| **B5** | **Infrarrojo cercano (NIR)** | **0.85–0.88 µm** | **30 m** | Alta reflectancia de hojas sanas; agua normalmente oscura |
| **B6** | **SWIR 1** | **1.57–1.65 µm** | **30 m** | Variaciones de humedad de la vegetación y del suelo |
| **B7** | **SWIR 2** | **2.11–2.29 µm** | **30 m** | Humedad, rocas, superficies quemadas y fuentes térmicas intensas |
| B8 | Pancromática | 0.50–0.68 µm | 15 m | Mayor detalle espacial en una imagen de banda ancha |
| B9 | Cirros | 1.36–1.38 µm | 30 m | Identificación de cirros elevados |
| **B10** | **Infrarrojo térmico (TIRS 1)** | **10.60–11.19 µm** | **100 m** | Radiación térmica emitida; base del producto ST_B10 |
| B11 | Infrarrojo térmico (TIRS 2) | 11.50–12.51 µm | 100 m | Información térmica complementaria |

Las bandas B2–B7 y el producto ST_B10 se usan directamente en este cuaderno. La banda térmica tiene **100 m de resolución nativa**, aunque los productos suelen distribuirse remuestreados a 30 m; el remuestreo no agrega detalle térmico real.

**Referencia:** [NASA – Spectral Bands and Applications](https://science.nasa.gov/mission/landsat/spectral-bands-and-applications/) · [USGS – Bandas Landsat](https://www.usgs.gov/faqs/what-are-band-designations-landsat-satellites) · [NASA – Interpretación de falsos colores](https://science.nasa.gov/earth/earth-observatory/how-to-interpret-a-false-color-satellite-image/).

In [ ]:

productos=W.SelectMultiple(options=list(PRESETS),value=tuple(PRESETS),description='Capas:',rows=15,layout=W.Layout(width='440px',height='310px'))
custom_r=W.Dropdown(options=list(BANDS),value='B7',description='R:')
custom_g=W.Dropdown(options=list(BANDS),value='B6',description='G:')
custom_b=W.Dropdown(options=list(BANDS),value='B2',description='B:')
formula=W.Text(value='(B7-B6)/(B7+B6+0.00001)',description='Fórmula:',layout=W.Layout(width='620px'))
# Fórmula inicial distinta de NDVI: contraste SWIR2-SWIR1, solo para exploración.
mascara=W.Checkbox(value=False,description='Ocultar QA inválido en RGB (no altera el contraste)')
percentil=W.FloatRangeSlider(value=(2,98),min=0,max=100,step=1,description='Contraste %:',layout=W.Layout(width='450px'))
procesar=W.Button(description='GENERAR GALERÍA COMPLETA EN 2 COLUMNAS',button_style='success',layout=W.Layout(width='380px'))
resultado=W.Output()
descargar_zip=W.Button(description='DESCARGAR ZIP GENERADO',button_style='info',layout=W.Layout(width='320px'))
ultimo_zip=None

def descargar_ultimo_zip(_):
    if not ultimo_zip or not Path(ultimo_zip).exists():
        print('Primero procesa una escena para generar el ZIP.')
        return
    try:
        from google.colab import files as colab_files
        colab_files.download(str(ultimo_zip))
    except ImportError:
        display(FileLink(str(ultimo_zip)))

descargar_zip.on_click(descargar_ultimo_zip)
display(W.HBox([productos,W.VBox([custom_r,custom_g,custom_b,formula,mascara,percentil])]),procesar,descargar_zip,resultado)


Button(button_style='success', description='GENERAR GALERÍA COMPLETA EN 2 COLUMNAS', layout=Layout(width='380p…

Button(button_style='info', description='DESCARGAR ZIP GENERADO', layout=Layout(width='320px'), style=ButtonSt…

Output()

### Representación cartográfica

Las composiciones RGB muestran la información espectral mediante un ajuste de contraste. El NDVI se representa en seis clases, dentro del intervalo −1 a +1, con tonos de marrón a verde. Los demás índices utilizan escalas continuas de −1 a +1. Las áreas sin datos válidos se indican en gris.

Los mapas se generan a partir de los archivos GeoTIFF y conservan sus coordenadas. Las áreas clasificadas como no aptas por QA pueden limitar la interpretación de índices y temperaturas.

## 4. Lectura y procesamiento

Las bandas se leen desde archivos Landsat georreferenciados y se llevan a una cuadrícula común en EPSG:4326. La reflectancia superficial se obtiene aplicando los factores de escala de Landsat Collection 2. La banda QA_PIXEL permite reconocer las condiciones de calidad registradas para cada píxel.

La escala espacial del recorte es aproximada de 30 m para las bandas ópticas. Esto no modifica la resolución nativa de las observaciones térmicas.

In [ ]:
def signed(item):return planetary_computer.sign(item)
def asset_href(item,keys):
    for k in keys:
        if k in item.assets:return item.assets[k].href,k
    raise KeyError(f'No hay asset {keys}; disponibles: {list(item.assets)}')
def grid(la,lo,rkm):
    bounds=bbox_area(la,lo,rkm)
    # 30m aprox; relación lon/lat adaptada a la latitud
    height=max(64,round((bounds[3]-bounds[1])*111320/30))
    width=max(64,round((bounds[2]-bounds[0])*111320*math.cos(math.radians(la))/30))
    return bounds,width,height,from_bounds(*bounds,width,height)
def read_asset(item,keys,shape,transform,resampling=Resampling.nearest):
    href,key=asset_href(item,keys)
    with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR',CPL_VSIL_CURL_ALLOWED_EXTENSIONS='.TIF,.tif',GDAL_HTTP_MAX_RETRY='3',GDAL_HTTP_RETRY_DELAY='2'):
        with rasterio.open(href) as src:
            with WarpedVRT(src,crs='EPSG:4326',transform=transform,width=shape[1],height=shape[0],resampling=resampling,nodata=src.nodata) as vrt:
                arr=vrt.read(1,masked=True)
    return arr,key
def sr(raw):
    a=np.asarray(raw,dtype='float32')*.0000275-.2
    a[np.ma.getmaskarray(raw)]=np.nan
    a[np.asarray(raw)==0]=np.nan
    a[(a < -0.2) | (a > 1.6)]=np.nan
    return a
def temp_c(raw):
    a=np.asarray(raw,dtype='float32')*.00341802+149.-273.15
    a[np.ma.getmaskarray(raw) | (np.asarray(raw)==0)]=np.nan
    return a
def qa_masks(qa):
    q=np.asarray(qa.filled(0),dtype='uint16');missing=np.ma.getmaskarray(qa)
    # QA_PIXEL bit 0 is fill; masked pixels are excluded. Do not infer cloud from missing data.
    def bit(i):return ((q>>i)&1).astype(bool)
    return {'fill':bit(0)|missing,'dilated_cloud':bit(1),'cirrus':bit(2),'cloud':bit(3),'cloud_shadow':bit(4),'snow':bit(5),'water':bit(7)}
def qa_classes(m):
    c=np.zeros(m['fill'].shape,dtype='uint8') # 0 clear
    c[m['water']]=1;c[m['snow']]=2;c[m['cloud_shadow']]=3
    c[m['cirrus']]=4;c[m['dilated_cloud']]=5;c[m['cloud']]=6;c[m['fill']]=255
    return c
def valid_analysis(m):return ~(m['fill']|m['dilated_cloud']|m['cirrus']|m['cloud']|m['cloud_shadow']|m['snow'])
def ratio(a,b):
    # Índices normalizados: descartar reflectancias negativas/no físicas y denominadores inestables.
    a=np.asarray(a,dtype='float32'); b=np.asarray(b,dtype='float32')
    good=np.isfinite(a)&np.isfinite(b)&(a>=0)&(b>=0)&((a+b)>0.02)
    out=np.full(a.shape,np.nan,dtype='float32')
    with np.errstate(divide='ignore',invalid='ignore'):
        np.divide(a-b,a+b,out=out,where=good)
    out[(out < -1.0001)|(out > 1.0001)]=np.nan
    return out
def safe_formula(expr,env):
    tree=ast.parse(expr,mode='eval')
    def walk(node):
        if isinstance(node,ast.Expression):return walk(node.body)
        if isinstance(node,ast.Name) and node.id in env:return env[node.id]
        if isinstance(node,ast.Constant) and isinstance(node.value,(int,float)):return float(node.value)
        if isinstance(node,ast.UnaryOp) and isinstance(node.op,(ast.UAdd,ast.USub)):
            v=walk(node.operand);return v if isinstance(node.op,ast.UAdd) else -v
        if isinstance(node,ast.BinOp) and isinstance(node.op,(ast.Add,ast.Sub,ast.Mult,ast.Div,ast.Pow)):
            a,b=walk(node.left),walk(node.right)
            with np.errstate(divide='ignore',invalid='ignore',over='ignore'):
                if isinstance(node.op,ast.Add):return a+b
                if isinstance(node.op,ast.Sub):return a-b
                if isinstance(node.op,ast.Mult):return a*b
                if isinstance(node.op,ast.Div):return a/b
                if isinstance(node.op,ast.Pow):return a**b
        raise ValueError('Expresión no permitida. Usa B2–B7, números, paréntesis y + - * / **')
    return walk(tree)
def stretch(rgb,pct=(2,98),mask=None):
    out=np.zeros_like(rgb,dtype='float32')
    for i in range(3):
        a=rgb[i];v=a[np.isfinite(a)&(mask if mask is not None else True)]
        if len(v)==0:continue
        lo,hi=np.percentile(v,pct)
        if hi<=lo:hi=lo+1e-5
        out[i]=np.clip((a-lo)/(hi-lo),0,1)
    return np.moveaxis(np.nan_to_num(out,nan=0),0,-1)
def geotiff(path,arr,transform,nodata=-9999):
    a=np.asarray(arr,dtype='float32')
    if a.ndim==2:a=a[None]
    with rasterio.open(path,'w',driver='GTiff',height=a.shape[1],width=a.shape[2],count=a.shape[0],dtype='float32',crs='EPSG:4326',transform=transform,nodata=nodata,compress='deflate',tiled=True) as dst:
        dst.write(np.where(np.isfinite(a),a,nodata).astype('float32'))
def geotiff_qa(path,a,transform):
    with rasterio.open(path,'w',driver='GTiff',height=a.shape[0],width=a.shape[1],count=1,dtype='uint8',crs='EPSG:4326',transform=transform,nodata=255,compress='deflate') as dst:dst.write(a.astype('uint8'),1)
print('Motor listo.')

Motor listo.


In [ ]:

# Visualización de los productos directamente desde GeoTIFF.
from rasterio.plot import plotting_extent
from matplotlib.colors import ListedColormap, BoundaryNorm
from PIL import Image

PALETAS = {
    'NDVI': ('ndvi', 'NDVI'),
    'NDMI': ('BrBG', 'NDMI'),
    'NDWI': ('RdYlBu', 'NDWI'),
    'NDSI': ('coolwarm', 'NDSI'),
    'custom_formula': ('viridis', 'Valor'),
    'temp': ('inferno', 'Temperatura superficial (°C)'),
}

# Paleta exacta de la prueba manual exitosa del usuario
NDVI_COLORES = ['#a94432', '#e6ad61', '#e8dd81', '#a9ce6a', '#4d9d53', '#086b3a']
NDVI_LIMITES = [-1, 0, 0.2, 0.4, 0.6, 0.8, 1]


def diagnosticar_array(nombre, array, maxpix=0):
    a = np.ma.asarray(array)
    vals = np.asarray(a.compressed(),dtype='float32')
    vals = vals[np.isfinite(vals)]
    if not vals.size:
        print(f'  {nombre}: SIN VALORES VALIDOS'); return False
    p2,p5,med,p95,p98 = np.percentile(vals,[2,5,50,95,98])
    std=float(np.std(vals))
    print(f'  {nombre}: n={len(vals)} | min={vals.min():.5f} p2={p2:.5f} p5={p5:.5f} med={med:.5f} p95={p95:.5f} p98={p98:.5f} max={vals.max():.5f} std={std:.5f}')
    if std < 1e-7 or (p98-p2) < 1e-6:
        print(f'  ADVERTENCIA: {nombre} tiene variación espacial casi nula')
        return False
    return True


def rgb_desde_tif(src, percentiles=(2,98), excluir_qa=None):
    # src.read() produce (3, alto, ancho); imshow RGB necesita (alto, ancho, 3)
    a = src.read(masked=True).astype('float32')
    if a.shape[0]!=3:
        raise ValueError(f'GeoTIFF RGB con forma incorrecta: {a.shape}')
    print('  RGB read-back shape:',a.shape,'->', (a.shape[1],a.shape[2],3))
    rgb = np.zeros((a.shape[1], a.shape[2], 3),dtype='float32')
    q_good = 0
    for k in range(3):
        ch = a[k].filled(np.nan)
        vals = ch[np.isfinite(ch)]  # QA NO participa en la normalización del contraste
        if len(vals) == 0:
            raise ValueError(f'Canal RGB {k+1} no tiene datos finitos')
        p_lo,p_hi = np.percentile(vals,percentiles)
        if p_hi <= p_lo:
            raise ValueError(f'Canal RGB {k+1} uniforme (p{percentiles[0]}={p_lo:.6f}, p{percentiles[1]}={p_hi:.6f})')
        diagnosticar_array(f'Canal {k+1}', a[k])
        rgb[:,:,k]=np.clip((ch-p_lo)/(p_hi-p_lo),0,1)
    rgb=np.nan_to_num(rgb,nan=0,posinf=1,neginf=0)
    if excluir_qa is not None and excluir_qa.shape == rgb.shape[:2] and np.mean(excluir_qa) >= 0.05:
        rgb[~excluir_qa] = 0.70
        print('  RGB: píxeles inválidos QA presentados en gris; NO intervienen en el contraste')
    # Test de salida en RGB, no confundir imagen uniforme real con procesamiento correcto
    if np.std(rgb,axis=(0,1)).max()<0.005:
        raise ValueError('RGB exportado resultaría plano: revisar banda, escena y percentiles.')
    return rgb


def render_desde_geotiff(tif, png, label, kind, fecha, qa_status='', rgb_mask=None, percentiles=(2,98)):
    with rasterio.open(tif) as src:
        extent = plotting_extent(src)  # extensión de coordenadas del ráster
        print('  CRS:',src.crs,'| GeoTIFF:',src.name,'| bounds:', tuple(round(v,5) for v in src.bounds))
        fig, ax = plt.subplots(figsize=(8,7))
        if kind in ('rgb','custom_rgb'):
            img = rgb_desde_tif(src,percentiles=percentiles,excluir_qa=rgb_mask)
            if label.startswith('Optimized natural color'):
                img = np.power(np.clip(img,0,1),0.86)  # realce de pantalla; GeoTIFF sin alterar
            ax.imshow(img,extent=extent,origin='upper',interpolation='nearest')
        elif kind == 'qa':
            a=src.read(1,masked=True)
            classes = np.ma.filled(a,255).astype('uint8')
            shown = np.where(classes==255,7,classes)
            qa_colors=['#377d22','#1768ac','#eeeeee','#444444','#e2bcff','#ffbd66','#ff5555','#000000']
            cm=ListedColormap(qa_colors)
            im=ax.imshow(shown,cmap=cm,vmin=-0.5,vmax=7.5,extent=extent,origin='upper',interpolation='nearest')
            cb=fig.colorbar(im,ax=ax,ticks=np.arange(8),shrink=0.78)
            cb.ax.set_yticklabels(['Libre','Agua','Nieve','Sombra','Cirros','Nube dilatada','Nube','Sin datos'])
            print('  Clases QA:',dict(zip(*np.unique(classes,return_counts=True))))
        else:
            # Abrir el ráster preservando su máscara de datos inválidos
            a=src.read(1,masked=True)
            a=np.ma.masked_invalid(a)
            diagnosticar_array(label,a)
            vals=a.compressed();vals=vals[np.isfinite(vals)]
            if vals.size == 0:
                plt.close(fig)
                raise ValueError('GeoTIFF sin píxeles válidos para visualizar')
            if kind=='index' and label=='NDVI':
                cmap=ListedColormap(NDVI_COLORES)
                cmap.set_bad('#d9d9d9')
                norm=BoundaryNorm(NDVI_LIMITES,cmap.N)
                im=ax.imshow(a,cmap=cmap,norm=norm,extent=extent,origin='upper',interpolation='nearest')
                cb=fig.colorbar(im,ax=ax,ticks=NDVI_LIMITES,shrink=0.85)
                cb.set_label('NDVI')
            else:
                if kind=='index':
                    key=label.split(' / ')[-1] if '/' in label else label.split(' (')[0]
                    if 'NDMI' in label: cmname='BrBG'; legend='NDMI'
                    elif 'NDWI' in label: cmname='RdYlBu'; legend='NDWI'
                    else: cmname='coolwarm';legend='NDSI'
                    cm=plt.get_cmap(cmname).copy();cm.set_bad('#d9d9d9')
                    im=ax.imshow(a,cmap=cm,vmin=-1,vmax=1,extent=extent,origin='upper',interpolation='nearest')
                else:
                    cm=plt.get_cmap('inferno' if kind=='temp' else 'viridis').copy()
                    cm.set_bad('#d9d9d9')
                    vmin,vmax=np.percentile(vals,[2,98])
                    if vmax<=vmin:
                        plt.close(fig)
                        raise ValueError('Valores uniformes; no generar una visualización plana')
                    im=ax.imshow(a,cmap=cm,vmin=vmin,vmax=vmax,extent=extent,origin='upper',interpolation='nearest')
                    legend='°C' if kind=='temp' else 'Valor'
                cb=fig.colorbar(im,ax=ax,shrink=0.85)
                cb.set_label(legend)
        ax.set_title(f'{label} | {fecha}{qa_status}',fontsize=11)
        ax.set_xlabel('Longitud (°)')
        ax.set_ylabel('Latitud (°)')
        ax.set_aspect(1/max(0.1,math.cos(math.radians((extent[2]+extent[3])/2))))
        fig.tight_layout()
        fig.savefig(png,dpi=140,bbox_inches='tight')
        plt.close(fig)
        return png


# Descripciones para las fichas de lectura espectral, basadas en NASA y USGS.
# Las fichas no cambian los valores GeoTIFF ni el cálculo de los índices.
from PIL import Image
import textwrap
import re

BAND_INFO = {
    'B2': ('Azul visible', '0.45–0.51 µm', 'Agua, bruma y reflectancia en el azul visible.'),
    'B3': ('Verde visible', '0.53–0.59 µm', 'Reflectancia verde de la vegetación y del agua.'),
    'B4': ('Rojo visible', '0.64–0.67 µm', 'Absorción de clorofila y diferencias suelo-vegetación.'),
    'B5': ('Infrarrojo cercano', '0.85–0.88 µm', 'Hojas vigorosas reflejan mucho; el agua suele ser oscura.'),
    'B6': ('SWIR 1', '1.57–1.65 µm', 'Variaciones de humedad de suelo y plantas.'),
    'B7': ('SWIR 2', '2.11–2.29 µm', 'Materiales secos, rocas y fuentes térmicas intensas.'),
    'B10': ('Infrarrojo térmico', '10.60–11.19 µm', 'Radiación térmica emitida por la superficie.'),
}


def texto_lectura(label):
    kind, bands = PRESETS[label]
    info = {'titulo': '', 'renglones': [], 'destaca': '', 'precaucion': ''}
    if kind in ('rgb', 'custom_rgb'):
        band_list = bands if kind == 'rgb' else [custom_r.value, custom_g.value, custom_b.value]
        info['titulo'] = 'Canales de visualización · R / G / B'
        info['renglones'] = [f'{canal} → {banda} · {BAND_INFO[banda][0]} · {BAND_INFO[banda][1]}: {BAND_INFO[banda][2]}'
                             for canal, banda in zip('RGB', band_list)]
        if label.startswith('True color'):
            info['destaca'] = 'Aspecto natural: vegetación verde, suelos expuestos, nubes y relieve.'
        elif label.startswith('Optimized'):
            info['destaca'] = 'Mismas B4/B3/B2 que True Color; gamma de visualización 0.86 para aclarar tonos.'
        elif label.startswith('False color (B5'):
            info['destaca'] = 'La vegetación vigorosa suele verse roja porque B5 (NIR) está asignada al canal R.'
        elif label.startswith('Wildfires'):
            info['destaca'] = 'Contraste de humedad y materiales; focos muy calientes pueden sobresalir en SWIR.'
            info['precaucion'] = 'Exactamente las mismas bandas que False color urban y SWIR. No detecta incendios automáticamente.'
        elif label.startswith('False color urban'):
            info['destaca'] = 'Contrastes de materiales, suelo desnudo y superficies urbanizadas, según el contexto.'
            info['precaucion'] = 'Misma combinación B7/B6/B4 que Wildfires / SWIR y SWIR.'
        elif label.startswith('SWIR (B7'):
            info['destaca'] = 'Diferencias entre superficies secas/húmedas y materiales geológicos; posible respuesta térmica intensa.'
            info['precaucion'] = 'Misma combinación B7/B6/B4 que las dos composiciones anteriores.'
        elif kind == 'custom_rgb':
            info['destaca'] = 'La interpretación depende de la asignación de bandas elegida por el estudiante.'
            info['precaucion'] = 'Los colores son artificiales si los canales reciben bandas infrarrojas.'
    elif kind == 'index':
        params = {
            'NDVI': ('(B5 − B4) / (B5 + B4)', ['B5', 'B4'], 'Valores elevados: cobertura vegetal con alta respuesta NIR; bajos: superficies desnudas, agua o vegetación escasa.'),
            'NDMI': ('(B5 − B6) / (B5 + B6)', ['B5', 'B6'], 'Respuesta relativa a humedad de la vegetación; no mide porcentaje de agua directamente.'),
            'NDWI': ('(B3 − B5) / (B3 + B5)', ['B3', 'B5'], 'Resalta posibles cuerpos de agua; sombras, edificios y rocas pueden confundirse.'),
            'NDSI': ('(B3 − B6) / (B3 + B6)', ['B3', 'B6'], 'Diferencia espectral para identificar posible nieve/hielo; requiere verificación.'),
        }
        ix = bands
        formula_ix, used, lectura = params[ix]
        info['titulo'] = 'Índice de una banda · color cartográfico (no RGB)'
        info['renglones'] = [f'Fórmula: {formula_ix}'] + [f'{b} · {BAND_INFO[b][0]} · {BAND_INFO[b][1]}' for b in used]
        info['destaca'] = lectura
        info['precaucion'] = 'Escala normalizada −1 a +1; gris = sin dato o píxel excluido. Validar QA.'
    elif kind == 'qa':
        info['titulo'] = 'Indicadores de calidad · sin bandas RGB ni longitud de onda'
        info['renglones'] = [
            'Fuente: QA_PIXEL, clasificación por bits (no reflectancia).',
            'Rojo: nube · gris oscuro: sombra · lila: cirros.',
            'Verde: libre en esta clasificación · azul: agua · negro: sin datos.'
        ]
        info['destaca'] = 'Detecta condiciones que pueden impedir la interpretación de los demás productos.'
        info['precaucion'] = 'Un mapa completamente rojo indica predominio del indicador «nube», no lava.'
    elif kind == 'temp':
        info['titulo'] = 'Producto térmico · una banda física, no RGB'
        info['renglones'] = [
            'ST_B10 · 10.60–11.19 µm · radiación infrarroja emitida.',
 'TIRS: 100 m nativos; producto distribuido remuestreado a 30 m.',
            'Leyenda en °C: temperatura estimada de superficie, no temperatura de lava.'
        ]
        info['destaca'] = 'Contrastes superficiales térmicos sujetos a atmósfera, emisividad, nubes y mezcla de píxeles.'
        info['precaucion'] = 'Con QA insuficiente, el mapa es exploratorio y no se debe interpretar cuantitativamente.'
    elif kind == 'custom_formula':
        used = sorted(set(re.findall(r'\bB[2-7]\b', formula.value)))
        info['titulo'] = 'Operación algebraica personalizada · resultado de una banda'
        info['renglones'] = [f'Expresión: {formula.value}'] + [f'{b} · {BAND_INFO[b][0]} · {BAND_INFO[b][1]}' for b in used[:3]]
        info['destaca'] = 'El mapa resalta diferencias relativas entre las bandas elegidas; su interpretación depende de la fórmula.'
        info['precaucion'] = 'La fórmula inicial contrasta B7 y B6 y NO es un detector térmico validado.'
    return info


def _wrap_text(s, width=99, maxlines=2):
    # Evita que una frase se salga de la ficha.
    lines = textwrap.wrap(s, width=width, break_long_words=False, break_on_hyphens=False)
    if len(lines) > maxlines:
        return '\n'.join(lines[:maxlines-1] + [lines[maxlines-1][:max(8,width-2)] + '…'])
    return '\n'.join(lines)


def ficha_didactica(label, mapa_png, ficha_png):
    """Crea una imagen independiente con un mapa PNG intacto y la leyenda pedagógica debajo."""
    guia = texto_lectura(label)
    fig = plt.figure(figsize=(8.5, 10.2), facecolor='white')
    gs = fig.add_gridspec(2, 1, height_ratios=[7.2, 2.55], hspace=0.015,
                          left=0.035, right=0.965, top=0.993, bottom=0.022)
    axm = fig.add_subplot(gs[0])
    with Image.open(mapa_png) as im:
        axm.imshow(im.convert('RGB'))
    axm.set_axis_off()
    axc = fig.add_subplot(gs[1])
    axc.set_xlim(0,1); axc.set_ylim(0,1); axc.set_axis_off()
    from matplotlib.patches import FancyBboxPatch, Rectangle
    axc.add_patch(FancyBboxPatch((0.01,0.025),0.98,0.94,boxstyle='round,pad=0.012',
                                facecolor='#F3F6F8',edgecolor='#C7D1D9',lw=0.85,
                                transform=axc.transAxes))
    axc.text(0.043,0.915,guia['titulo'],fontsize=11.2,weight='bold',va='top',
             color='#18364C',transform=axc.transAxes)
    y = 0.755
    if PRESETS[label][0] in ('rgb','custom_rgb'):
        for i, linea in enumerate(guia['renglones']):
            color = ['#d14c4c','#3d9d60','#4275c5'][i]
            axc.add_patch(Rectangle((.044,y-.026),.022,.055,color=color,transform=axc.transAxes))
            axc.text(.081,y,_wrap_text(linea,99,1),fontsize=8.65,va='center',
                     color='#253342',transform=axc.transAxes)
            y -= .135
    else:
        for linea in guia['renglones'][:4]:
            axc.text(.045,y,_wrap_text(linea,102,1),fontsize=9.3,va='center',
                     color='#253342',transform=axc.transAxes)
            y -= .133
    axc.text(.044,0.267,_wrap_text('Se observa: '+guia['destaca'],100,2),fontsize=9.0,
             weight='medium',color='#22384B',va='top',transform=axc.transAxes)
    if guia['precaucion']:
        axc.text(.044,0.125,_wrap_text('Nota: '+guia['precaucion'],100,2),fontsize=8.5,
                 color='#6A4A30',va='top',transform=axc.transAxes)
    fig.savefig(ficha_png,dpi=132,facecolor='white')
    plt.close(fig)
    return ficha_png


def galeria_dos_columnas(cards,out_png):
    """Conserva los mapas originales y añade una ficha visible debajo de cada mapa."""
    if not cards:
        raise RuntimeError('No hay PNG válidos para la galería')
    fichas=[]
    for label,png in cards:
        p=Path(png)
        f=p.with_name(p.stem+'_FICHA.png')
        ficha_didactica(label,p,f)
        fichas.append(f)
    imgs=[]
    try:
        for p in fichas:
            with Image.open(p) as im:
                imgs.append(im.convert('RGB'))
        iw=max(im.width for im in imgs)
        ih=max(im.height for im in imgs)
        gap=24
        rows=math.ceil(len(imgs)/2)
        canvas=Image.new('RGB',(2*iw+3*gap,rows*ih+(rows+1)*gap),'#ECF1F4')
        for i,im in enumerate(imgs):
            x=gap+(i%2)*(iw+gap); y=gap+(i//2)*(ih+gap)
            canvas.paste(im,(x,y))
        canvas.save(out_png,optimize=True)
        display(canvas)
    finally:
        for im in imgs: im.close()
    return fichas

print('Visualización cartográfica preparada.')


Visualización cartográfica preparada.


## 5. Diagnóstico y galería

El procesamiento presenta el identificador de la escena, los valores QA_PIXEL más frecuentes, el porcentaje de píxeles que cumplen los criterios de calidad y los estadísticos de las bandas e índices. Debajo de cada mapa se presenta una ficha con la asignación espectral, sus longitudes de onda y la lectura principal. Cada capa se genera en GeoTIFF y PNG. Al finalizar, se muestra una galería de dos columnas.

### Interpretación de QA_PIXEL

**QA_PIXEL es una capa de calidad, no una clasificación de uso o cobertura del suelo.** El mapa distingue píxeles señalados como nube, nube dilatada, cirros, sombra, nieve, agua o libres de las condiciones anteriores. Los indicadores pueden superponerse; para el mapa se aplica una prioridad de clases.

Si casi toda el área aparece roja, significa que la mayoría de sus píxeles quedaron clasificados como **nube**. Por ejemplo, para la escena Landsat 8 del **2 de agosto de 2026** sobre Fuego, el diagnóstico mostró aproximadamente **99.99 % de píxeles con indicador de nube**. En ese caso, los índices y la temperatura deben interpretarse únicamente como resultados exploratorios.

Para comparar, la escena Landsat 9 del **10 de agosto de 2026** presentó aproximadamente **96.2 % de píxeles aptos para el análisis** en el recorte utilizado. Estos porcentajes corresponden a las pruebas de esa área y no a toda Guatemala.

### Ejecución

1. Buscar escenas y seleccionar la adquisición de interés.
2. Elegir las capas o conservar la selección completa.
3. Pulsar **GENERAR GALERÍA COMPLETA EN 2 COLUMNAS**.
4. Revisar el diagnóstico QA, las imágenes y los textos explicativos que aparecen debajo.
5. Pulsar **DESCARGAR ZIP GENERADO** para obtener los productos.

Para una primera comparación puede utilizarse la escena Landsat 9 del **10 de agosto de 2026** sobre el volcán de Fuego.

In [ ]:

# Guardar cada producto en GeoTIFF y crear el PNG desde el archivo georreferenciado.
def ejecutar_proceso(_):
    global ultimo_zip
    with resultado:
        clear_output(wait=True)
        try:
            assert escenas and selector.value,'Primero busca y selecciona una escena'
            item0=next((x for x in escenas if x.id==selector.value),None)
            assert item0 is not None,'La escena elegida no coincide con el catálogo'
            fecha=item0.datetime.strftime('%Y-%m-%d')
            print('ESCENA EXACTA:',item0.id,'|',item0.datetime.isoformat())
            item=signed(item0)
            bounds,w,h,transform=grid(lat.value,lon.value,radio.value)
            shape=(h,w);cache={};fallos=[];archivos=[];cards=[];estados={}
            folder=OUT/item.id
            folder.mkdir(parents=True,exist_ok=True)

            def get_band(name):
                if name not in cache:
                    raw,_=read_asset(item,[BANDS[name]],shape,transform,Resampling.bilinear)
                    cache[name]=sr(raw)
                    diagnosticar_array(name+' reflectancia',np.ma.masked_invalid(cache[name]))
                return cache[name]

            qa,_=read_asset(item,['qa_pixel'],shape,transform,Resampling.nearest)
            m=qa_masks(qa);qa_valid=valid_analysis(m)
            qvalues=np.asarray(qa.filled(0),dtype=np.uint16)
            uq,cts=np.unique(qvalues,return_counts=True)
            print('QA_PIXEL valores frecuentes:',sorted(zip(uq.tolist(),cts.tolist()),key=lambda x:-x[1])[:12])
            print('QA_PIXEL píxeles enmascarados: %.2f%%' % (100*np.ma.getmaskarray(qa).mean()))
            print('QA_PIXEL bits (%):',{k:round(100*float(v.mean()),2) for k,v in m.items()})
            qa_pct=100*float(qa_valid.mean())
            print('QA aptos en recorte: %.3f%%' % qa_pct)
            # Resumen adicional para la lectura de QA_PIXEL en clase.
            qa_clases = qa_classes(m)
            nombres_qa = {0:'Libre',1:'Agua',2:'Nieve',3:'Sombra',4:'Cirros',
                          5:'Nube dilatada',6:'Nube',255:'Sin datos'}
            resumen_qa = {nombre: round(100.0*float(np.mean(qa_clases==codigo)), 2)
                          for codigo,nombre in nombres_qa.items()}
            print('Clasificación de calidad del recorte (%):', resumen_qa)
            if resumen_qa['Nube'] >= 95:
                print('AVISO QA: predominan los píxeles clasificados como nube; '
                      'la capa de calidad aparecerá casi toda roja.')

            # Si quedan solo unos pocos píxeles aislados, NO usarlos para estirar todo el RGB.
            # El límite del 5% determina si se utiliza la máscara de calidad al visualizar índices;
            # no es un umbral oficial de calidad USGS ni significa validación científica.
            qa_suficiente_para_mapa = qa_pct >= 5.0
            if not qa_suficiente_para_mapa:
                print('ADVERTENCIA: QA insuficiente para visualizar índice enmascarado: índices EXPLORATORIOS')

            for label in productos.value:
                kind,recipe=PRESETS[label]
                slug=''.join(ch if ch.isalnum() else '_' for ch in label).strip('_')[:52]
                stem=f'{item.id}_{slug}'
                tif=folder/(stem+'.tif');png=folder/(stem+'.png')
                print('\n'+'='*75+'\nPRODUCTO:',label)
                try:
                    qa_status=''
                    if kind in ('rgb','custom_rgb'):
                        bands=recipe if kind=='rgb' else [custom_r.value,custom_g.value,custom_b.value]
                        data=np.stack([get_band(b) for b in bands],axis=0)
                        geotiff(tif,data,transform)
                        # El contraste RGB se calcula sin usar los indicadores QA.
                        mask_rgb=qa_valid if mascara.value and qa_suficiente_para_mapa else None
                        if mascara.value and not qa_suficiente_para_mapa:
                            print('RGB: se omite ocultamiento QA por cobertura insuficiente')
                        render_desde_geotiff(tif,png,label,kind,fecha,rgb_mask=mask_rgb,percentiles=percentil.value)
                    elif kind=='qa':
                        data=qa_classes(m)
                        geotiff_qa(tif,data,transform)
                        render_desde_geotiff(tif,png,label,kind,fecha)
                    else:
                        if kind=='index':
                            a,b={'NDVI':('B5','B4'),'NDMI':('B5','B6'),'NDWI':('B3','B5'),'NDSI':('B3','B6')}[recipe]
                            data=ratio(get_band(a),get_band(b))
                            if recipe=='NDVI':
                                print('NDVI basado en B5 y B4 convertidos a reflectancia Collection 2')
                        elif kind=='custom_formula':
                            import re
                            needed=set(re.findall(r'\bB[2-7]\b',formula.value))
                            data=np.asarray(safe_formula(formula.value,{k:get_band(k) for k in needed}),dtype='float32')
                            if data.ndim==0:data=np.full(shape,float(data),dtype='float32')
                            assert data.shape==shape,'La fórmula no produjo un ráster del tamaño esperado'
                        elif kind=='temp':
                            raw,_=read_asset(item,['lwir11','st','ST_B10'],shape,transform,Resampling.bilinear)
                            data=temp_c(raw)
                        else:
                            raise ValueError('Tipo de producto no soportado: '+kind)
                        data=np.asarray(data,dtype='float32')
                        if qa_suficiente_para_mapa:
                            data=np.where(qa_valid,data,np.nan).astype('float32')
                        else:
                            qa_status=' | QA NO VALIDADO — EXPLORATORIO'
                        if kind=='index':
                            data=np.where(np.isfinite(data)&(data>=-1)&(data<=1),data,np.nan)
                        geotiff(tif,data,transform)
                        render_desde_geotiff(tif,png,label,kind,fecha,qa_status=qa_status)
                    archivos.extend([tif,png]);cards.append((label,png));estados[label]='EXPLORATORIO' if qa_status else 'GENERADO'
                except Exception as e:
                    fallos.append((label,repr(e)));estados[label]='ERROR: '+repr(e)
                    print('ERROR EN CAPA',label,':',repr(e))
                    if tif.exists() and not png.exists():
                        print('El GeoTIFF quedó guardado para diagnóstico: ',tif.name)
                        archivos.append(tif)

            info={'scene_id':item.id,'fecha_UTC':item.datetime.isoformat(),
                  'satellite':item.properties.get('platform'), 'center_lat':lat.value,'center_lon':lon.value,
                  'radius_km':radio.value,'bounds_EPSG4326':list(bounds),
                  'nubes_escena_pct':item.properties.get('eo:cloud_cover'),
                  'qa_validos_area_pct':round(qa_pct,4),
                  'qa_por_bit_pct':{k:round(100*float(v.mean()),3) for k,v in m.items()},
                  'guias_por_producto':{label:texto_lectura(label) for label,_ in cards},
                  'estados':estados,'fallos':fallos,'nota':'Índices con QA insuficiente son exploratorios, no validación científica'}
            meta=folder/(item.id+'_metadatos.json')
            meta.write_text(json.dumps(info,ensure_ascii=False,indent=2),encoding='utf-8')
            archivos.append(meta)
            if cards:
                galeria=folder/(item.id+'_GALERIA_2_COLUMNAS.png')
                fichas=galeria_dos_columnas(cards,galeria)
                archivos.extend(fichas)
                archivos.append(galeria)
            else:
                print('No se generaron PNG válidos; consulta diagnóstico y errores')

            zip_path=folder/(item.id+'_PRODUCTOS_LANDSAT.zip')
            with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as zf:
                for p in archivos:
                    if p.exists():zf.write(p,arcname=p.name)
            ultimo_zip=zip_path
            print('\nGENERADOS:',len(cards),'CAPAS | FALLOS:',len(fallos))
            for label,msg in fallos:print('FALLÓ:',label,msg)
            print('ZIP:',zip_path)
            display(FileLink(str(zip_path),result_html_prefix='ZIP de todos los productos: '))
            print('Puedes usar el botón DESCARGAR ZIP GENERADO para iniciar descarga en Colab.')
        except Exception as e:
            print('ERROR GENERAL:',repr(e));raise

procesar.on_click(ejecutar_proceso)
print('Selecciona una escena y pulsa GENERAR GALERÍA COMPLETA EN 2 COLUMNAS.')


Selecciona una escena y pulsa GENERAR GALERÍA COMPLETA EN 2 COLUMNAS.


## 6. Archivos de salida y referencias

El archivo ZIP contiene los **GeoTIFF originales** (reflectancias, índices o QA), los **PNG cartográficos**, las **fichas didácticas PNG** con información debajo de cada imagen, la **galería completa en dos columnas** y un archivo **JSON** con los metadatos de adquisición y las descripciones de las bandas.

Los índices son cálculos radiométricos, no clasificaciones completas de la superficie. Los colores aplicados a NDVI, NDMI, NDWI, NDSI y ST_B10 son paletas de visualización. Los productos con QA insuficiente se marcan como exploratorios.

### Consideraciones para interpretación

- Una composición SWIR puede ayudar a identificar respuestas compatibles con una superficie muy caliente, pero **no es un detector validado de lava o incendios**.
- `QA_PIXEL` identifica condiciones como nubes, cirros y sombras; **no equivale al producto SCL de Sentinel-2** ni clasifica materiales volcánicos.
- La temperatura superficial `ST_B10` proviene de un producto térmico con correcciones atmosféricas y emisividad; no es igual a temperatura de lava. El sensor térmico tiene 100 m de resolución nativa.
- **NDWI tiene distintas definiciones en la literatura.** Aquí se usa `Green–NIR` (B3–B5) para realzar agua superficial; **NDMI** usa `NIR–SWIR1` (B5–B6) para humedad de la vegetación.
- Si dos composiciones comparten la asignación R/G/B y el estiramiento, mostrarán la misma imagen; un nombre distinto no modifica los datos.

### Fuentes consultadas

- [NASA — Spectral Bands and Applications](https://science.nasa.gov/mission/landsat/spectral-bands-and-applications/)
- [NASA — How to Interpret a False-Color Satellite Image](https://science.nasa.gov/earth/earth-observatory/how-to-interpret-a-false-color-satellite-image/)
- [USGS — Landsat Quality Assessment Bands](https://www.usgs.gov/landsat-missions/landsat-collection-2-quality-assessment-bands)
- [USGS — Landsat Collection 2 Surface Temperature](https://www.usgs.gov/landsat-missions/landsat-collection-2-surface-temperature)
- [Esri — Band Combinations for Landsat 8](https://www.esri.com/arcgis-blog/products/product/imagery/band-combinations-for-landsat-8)
- [USGS — Normalized Difference Moisture Index](https://www.usgs.gov/landsat-missions/normalized-difference-moisture-index)